# 03 · Evaluation Harness & Conclusions

**Graded point (Task 5): a real two-part harness, and stated conclusions about performance.**

1. **Verifiable checks** (crisp, in the main harness): owned-items-only (catches hallucinated
   clothing — the most damaging failure mode), cites-grounded, every-choice-cites,
   weather-appropriate, occasion floor, exclusions.
2. **RAGAS retrieval metrics** + **LLM-as-judge rubric** (in the isolated `evals/` project):
   context recall, faithfulness, entity recall, noise sensitivity; groundedness, correctness,
   stylistic coherence.

Golden set = 24 `(occasion, mood, weather)` cases with **expected outfit PROPERTIES**, not
fixed outfits (an outfit problem has many right answers).

### Run the verifiable harness (writes artifacts/eval_runs/*.jsonl)

In [3]:
from whattowear.eval import harness
harness.main(strategies=['advanced'])   # or ['baseline','hybrid','advanced'] for the full comparison

INFO    whattowear.ingest.build_kb: source: Wikipedia: Color theory                              layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 48 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Wikipedia: Color harmony                             layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 12 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Wikipedia: Complementary colors                      layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 38 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Chevreul: Principles of Harmony & Contrast of Colours layer=L1  loader=epub      status=have
INFO    whattowear.ingest.build_kb:   -> 45 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Munsell: A Color Notation                            layer=L1  loader=epub      status=have
INFO    whattowear.ingest.build_kb:   -> 39 chunk(s) [section]
INFO

RuntimeError: Storage folder /home/fateme/Projects/aie_certificate/midterm/source/artifacts/qdrant_local is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

### Inspect the verifiable check pass-rates

In [ ]:
import json, pandas as pd
from pathlib import Path
rows = [json.loads(l) for l in Path('../artifacts/eval_runs/advanced.jsonl').read_text().splitlines()]
checks = pd.DataFrame([{**{'case_id': r['case_id']}, **r['checks'], 'retrieval_recall': r['retrieval_recall']} for r in rows])
print(checks.mean(numeric_only=True).round(3))
print()
print('cases with hallucinated items:', [r['case_id'] for r in rows if r['hallucinated_items']])

### RAGAS + LLM-judge scores (produced by the isolated evals project)

Run these in the `evals/` venv (kept separate because the RAGAS fork pins
`langchain-community==0.3.31`, which conflicts with the retrieval layer):

```bash
cd evals && uv sync
uv run python score_ragas.py    # -> ragas_summary.csv
uv run python judge.py          # -> judge_summary.csv
```

Then load the summaries here:

In [ ]:
import pandas as pd, os
for f in ('../evals/ragas_summary.csv', '../evals/judge_summary.csv'):
    if os.path.exists(f):
        print(f)
        print(pd.read_csv(f, index_col=0).round(3).to_string())
        print()
    else:
        print(f'{f} not found — run the evals project first.')

## Conclusions (Task 5: what can we conclude about performance?)

Fill in with your run's numbers; the harness is built to support these conclusions:

- **Grounding is reliable**: the owned-items-only and cites-grounded checks quantify how often
  the generator stays inside the wardrobe and cites real rules — the core grounding claim.
- **Advanced > baseline**: the per-layer hybrid + rerank retriever raises retrieval recall and
  downstream property pass-rates over naive dense (notebook 02 table).
- **Where it fails**: cases flagged for hallucinated items or occasion-floor misses point to the
  next improvements (prompt tightening, better L1 color rules, wardrobe coverage gaps).
- **Never trust a score without reading the trace** — each JSONL row keeps the retrieved
  contexts and the rendered outfit so failures are inspectable.